## Text Generation Model

## Import Libraries

In [18]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LSTM, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping

## Data Gathering

In [19]:
with open("/content/output.txt", "r", encoding="utf-8") as file:
    text = file.read()

print(text[:500])

I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.
"Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually small sized unsalted. Not sure if this was an error or if the vendor intended to represent the product as ""Jumbo""."
"This is a confection that has been aroun


## Tokenization

['this','is','to','explain','this','concept','to','students']

this : 2
is : 1
to :  2
explain : 1
concept : 1
students: 1

highest frequency count will be given index 1, followed by 2nd highest

this : 1, to:2, is: 3, explain : 4,concept:5,students:6

In [20]:
text=text[:60000]
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words=len(tokenizer.word_index)+1
print("Vocabulary Size:",total_words)

# Reverse mapping : index -> word
index_word = {i : word for word,i in tokenizer.word_index.items()}


Vocabulary Size: 2085


In [21]:
tokenizer.word_index

{'the': 1,
 'i': 2,
 'and': 3,
 'a': 4,
 'to': 5,
 'it': 6,
 'of': 7,
 'is': 8,
 'this': 9,
 'for': 10,
 'in': 11,
 'my': 12,
 'br': 13,
 'with': 14,
 'have': 15,
 'was': 16,
 'but': 17,
 'food': 18,
 'that': 19,
 'you': 20,
 'on': 21,
 'are': 22,
 'so': 23,
 'not': 24,
 'as': 25,
 'good': 26,
 'they': 27,
 'these': 28,
 'like': 29,
 'at': 30,
 'them': 31,
 'all': 32,
 'great': 33,
 'be': 34,
 'one': 35,
 'or': 36,
 'can': 37,
 'we': 38,
 "it's": 39,
 'has': 40,
 'product': 41,
 'very': 42,
 'than': 43,
 'if': 44,
 'love': 45,
 'taste': 46,
 'just': 47,
 'other': 48,
 'dog': 49,
 'too': 50,
 'me': 51,
 'were': 52,
 'really': 53,
 'flavor': 54,
 'when': 55,
 'oatmeal': 56,
 'had': 57,
 'will': 58,
 'from': 59,
 'more': 60,
 'little': 61,
 'your': 62,
 'much': 63,
 'amazon': 64,
 'better': 65,
 'sugar': 66,
 'now': 67,
 "i'm": 68,
 "don't": 69,
 'up': 70,
 'use': 71,
 "i've": 72,
 'time': 73,
 'tea': 74,
 'only': 75,
 'our': 76,
 'no': 77,
 'eat': 78,
 'about': 79,
 'instant': 80,
 'pric

In [22]:
index_word

{1: 'the',
 2: 'i',
 3: 'and',
 4: 'a',
 5: 'to',
 6: 'it',
 7: 'of',
 8: 'is',
 9: 'this',
 10: 'for',
 11: 'in',
 12: 'my',
 13: 'br',
 14: 'with',
 15: 'have',
 16: 'was',
 17: 'but',
 18: 'food',
 19: 'that',
 20: 'you',
 21: 'on',
 22: 'are',
 23: 'so',
 24: 'not',
 25: 'as',
 26: 'good',
 27: 'they',
 28: 'these',
 29: 'like',
 30: 'at',
 31: 'them',
 32: 'all',
 33: 'great',
 34: 'be',
 35: 'one',
 36: 'or',
 37: 'can',
 38: 'we',
 39: "it's",
 40: 'has',
 41: 'product',
 42: 'very',
 43: 'than',
 44: 'if',
 45: 'love',
 46: 'taste',
 47: 'just',
 48: 'other',
 49: 'dog',
 50: 'too',
 51: 'me',
 52: 'were',
 53: 'really',
 54: 'flavor',
 55: 'when',
 56: 'oatmeal',
 57: 'had',
 58: 'will',
 59: 'from',
 60: 'more',
 61: 'little',
 62: 'your',
 63: 'much',
 64: 'amazon',
 65: 'better',
 66: 'sugar',
 67: 'now',
 68: "i'm",
 69: "don't",
 70: 'up',
 71: 'use',
 72: "i've",
 73: 'time',
 74: 'tea',
 75: 'only',
 76: 'our',
 77: 'no',
 78: 'eat',
 79: 'about',
 80: 'instant',
 81: '

## Create input sequences: Instead of splitting line by line, use continuous text for better learning

In [24]:
text[:10]

'I have bou'

In [25]:
token_list = tokenizer.texts_to_sequences([text])[0]
input_sequences=[]

# Create sequences of 6 words
# First 5 words = input , 6th word =target
for i in range(5,len(token_list)):
  n_gram_sequence = token_list[i-5:i+1]
  input_sequences.append(n_gram_sequence)

print(input_sequences[:5])

[[2, 15, 122, 196, 7, 1], [15, 122, 196, 7, 1, 964], [122, 196, 7, 1, 964, 485], [196, 7, 1, 964, 485, 49], [7, 1, 964, 485, 49, 18]]


In [26]:
token_list

[2,
 15,
 122,
 196,
 7,
 1,
 964,
 485,
 49,
 18,
 197,
 3,
 15,
 84,
 31,
 32,
 5,
 34,
 7,
 26,
 115,
 1,
 41,
 628,
 60,
 29,
 4,
 965,
 43,
 4,
 966,
 967,
 3,
 6,
 486,
 65,
 12,
 968,
 8,
 487,
 3,
 92,
 969,
 9,
 41,
 65,
 43,
 215,
 41,
 284,
 970,
 25,
 629,
 971,
 245,
 1,
 245,
 52,
 216,
 246,
 630,
 972,
 24,
 153,
 44,
 9,
 16,
 107,
 973,
 36,
 44,
 1,
 488,
 974,
 5,
 975,
 1,
 41,
 25,
 629,
 9,
 8,
 4,
 976,
 19,
 40,
 85,
 330,
 4,
 331,
 977,
 6,
 8,
 4,
 978,
 979,
 980,
 981,
 14,
 631,
 11,
 9,
 247,
 982,
 3,
 6,
 8,
 285,
 248,
 286,
 983,
 3,
 171,
 984,
 985,
 14,
 632,
 66,
 3,
 6,
 8,
 4,
 286,
 986,
 7,
 987,
 24,
 50,
 387,
 3,
 42,
 988,
 2,
 287,
 131,
 9,
 288,
 289,
 44,
 20,
 22,
 489,
 14,
 1,
 989,
 7,
 990,
 249,
 991,
 1,
 992,
 1,
 633,
 3,
 1,
 993,
 9,
 8,
 1,
 289,
 19,
 994,
 995,
 248,
 490,
 116,
 154,
 634,
 3,
 996,
 5,
 1,
 633,
 44,
 20,
 22,
 172,
 10,
 1,
 997,
 491,
 11,
 998,
 2,
 332,
 2,
 15,
 84,
 6,
 2,
 108,
 9,
 11,
 999,
 5

## Padding
## Split X and Y

In [27]:
max_sequence_length =6
input_sequences = np.array(input_sequences)
X = input_sequences[:, :-1] # all sequences except last word/index
y = input_sequences[:, -1] # only last word/index

'''
[7, 121, 2, 1, 634, 158]
x= [7, 121, 2, 1, 634 ]
y = [158]
'''

# Convert target to one-hot encoding
y = to_categorical(y, num_classes=total_words)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (11175, 5)
y shape: (11175, 2085)


## Build LSTM Model

In [28]:
model = Sequential()
model.add(Input(shape=(max_sequence_length-1,)))
model.add(Embedding(input_dim=total_words,output_dim=128))
model.add(LSTM(150,return_sequences=True,dropout=0.2))
model.add(LSTM(100,dropout=0.2))
model.add(Dense(100,activation='relu'))
model.add(Dense(total_words,activation='softmax'))
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 5, 128)         │       266,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 5, 150)         │       167,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2085)           │       210,585 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 755,365 (2.88 MB)

 Trainable params: 755,365 (2.88 MB)

 Non-trainable params: 0 (0.00 B)

## Model Training

In [29]:
early_stop = EarlyStopping(monitor='loss',patience=3,restore_best_weights=True)

history = model.fit(X,y,epochs=30,batch_size=32,verbose=1,callbacks=[early_stop])

Epoch 1/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.0417 - loss: 6.5983
Epoch 2/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.0423 - loss: 6.2222
Epoch 3/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.0415 - loss: 6.1057
Epoch 4/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.0467 - loss: 5.9714
Epoch 5/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.0502 - loss: 5.7791
Epoch 6/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.0532 - loss: 5.6091
Epoch 7/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.0596 - loss: 5.4684
Epoch 8/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.0647 - loss: 5.3197
Epoch 9/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.0714 - loss: 5.1810
Epoch 10/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.0819 - loss: 5.0360
Epoch 11/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.0911 - loss: 4.8786
Epoch 12/30
350/350 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/s

## Using the trained model for final predictions
## TEMPERATURE + TOP-K SAMPLING
Temperature effect
0.5 → safer predictions
0.8 → balanced
1.2 → more creative/random

In [30]:
model.save("TextGenerationModel.keras")

In [31]:
def sample_with_temperature(preds, temperature=0.8, top_k=5):
    preds = np.asarray(preds).astype("float64")
    # [0.87,0.09,0.56,0.44,0.37,.............,0.89,0.32,...]

    # Select top k probabilities
    top_indices = np.argsort(preds)[-top_k:]
    # argsort : [0.09,0.32,0.37,0.44,0.56,0.87,0.89,........]
    # index of top k proabilities : [index]
    top_probs = preds[top_indices]
    #top k probs

    # Apply temperature scaling
    top_probs = np.log(top_probs + 1e-10) / temperature
    exp_probs = np.exp(top_probs)
    top_probs = exp_probs / np.sum(exp_probs)

    return np.random.choice(top_indices, p=top_probs)

## Text Generation method

In [32]:
def generate_text(seed_text, next_words=20):
    output_text = seed_text
    generated_words = []

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([output_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_length - 1,
            padding='pre'
        ) # [0,0,0,189,45]

        predicted_probs = model.predict(token_list, verbose=0)[0] # [[0.89,0.07,.....,]] = [0.89,0.07,.....,]

        predicted_index = sample_with_temperature(
            predicted_probs,
            temperature=0.8,
            top_k=5
        )

        next_word = index_word.get(predicted_index, "")

        # avoid immediate repetition
        if next_word in generated_words[-3:]:
            continue

        generated_words.append(next_word)
        output_text += " " + next_word

    return output_text

## Predictions

In [34]:
print(generate_text("tasty", next_words=15))

tasty weight br i stopped it these much to last its than a chance of wow
